# Router-Only Salvage: `Clustering_V0_Full_k2` + `Clustering_Backbone54_k2` on ECE
## Freeze experts, fix routing — ECE strictly unseen

The two static-routed MoE models fail on ECE (`RMSE 0.100 / 0.144`, bias `+0.07 / +0.13`)
because the static KMeans router sends `Lost_Meadow (100%)` and `Renton_Home (90%)`
to the wet-mountain expert, while dynamic routers send 100% of ECE rows to the dry
expert. This notebook reuses the SAME frozen experts (fit on WA `trainval` only) and
compares inference-time routing overrides: `as_routed`, `c0_only`, `c1_only`,
`gapi_transplant`, `dynamic_transplant`, `seasonal`, and `margin_fallback`
(ambiguous static rows fall back to the `Global_Single_54` expert).
ECE targets are used for evaluation only. Thresholds come from WA `trainval` only.


## 1. Load the versioned experiment implementation
The notebook imports the tracked runner instead of duplicating logic, so results
reproduce from a clean checkout. Paths resolve from the repository root.


In [1]:
from pathlib import Path
import importlib.util
import sys

cur = Path.cwd().resolve()
while cur != cur.parent:
    if (cur / "data" / "splits").exists() and (cur / "notebooks").exists():
        PROJECT_ROOT = cur
        break
    cur = cur.parent

EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.4-ece-router-salvage-1.0"
sys.path.insert(0, str(EXP_DIR))
RUNNER_PATH = EXP_DIR / "run_salvage.py"
spec = importlib.util.spec_from_file_location("router_salvage", RUNNER_PATH)
salvage = importlib.util.module_from_spec(spec)
spec.loader.exec_module(salvage)
config = salvage.load_configuration()
print(f"Experiment: {config['experiment']['name']}")
print(f"Families: {config['families']}")
print(f"Policies: {config['policies']}")
print(f"Seeds: {config['seeds']}")
print(f"Constraint: {config['experiment']['constraint']}")


Experiment: derived_8.4-ece-router-salvage-1.0
Families: ['Clustering_V0_Full_k2', 'Clustering_Backbone54_k2']
Policies: ['as_routed', 'c0_only', 'c1_only', 'gapi_transplant', 'dynamic_transplant', 'seasonal', 'margin_fallback']
Seeds: [42, 7, 13, 101, 123]
Constraint: ECE strictly unseen; experts fit on WA trainval only; routing is the only inference-time change.


## 2. Run the router-only salvage
Fits routers and experts on WA `trainval` only, then scores ECE (`zero` and
`native-missing` inputs) under each routing policy. Per-seed checkpoints resume,
so re-execution is fast when artifacts exist. From a clean checkout this cell
trains (documented timeout 3600s, CPU).


In [2]:
salvage.main([])  # [] = defaults; avoids parsing the kernel's argv


WA margin p5: v0=1.9560 backbone=1.5481
train_rows=14608 ece_rows=150 features=54 families=['Clustering_V0_Full_k2', 'Clustering_Backbone54_k2']
seed=42 resumed (28 rows)
seed=7 resumed (28 rows)
seed=13 resumed (28 rows)
seed=101 resumed (28 rows)
seed=123 resumed (28 rows)



ROUTER SALVAGE SUMMARY (mean over seeds)
                  family ece_input             policy  rmse_mean  rmse_std  mae_mean  bias_mean  ubrmse_mean    r2_mean  pearson_mean  rmse_change_vs_as_routed
   Clustering_V0_Full_k2      zero          as_routed   0.117805  0.001259  0.093832   0.070703     0.094226  -5.269953     -0.486731                  0.000000
   Clustering_V0_Full_k2      zero            c0_only   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965      0.141216                 -0.066460
   Clustering_V0_Full_k2      zero            c1_only   0.155617  0.001981  0.148323   0.148323     0.047076  -9.941304      0.132800                  0.037812
   Clustering_V0_Full_k2      zero    gapi_transplant   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965      0.141216                 -0.066460
   Clustering_V0_Full_k2      zero dynamic_transplant   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965      0.141216                 -0.06646

## 3. Routing audit and policy comparison
Checks how many ECE rows each policy redirects, the WA-only margin thresholds,
and the pooled RMSE / bias recovery versus `as_routed`.


In [3]:
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

summary = pd.read_csv(EXP_DIR / "summary.csv")
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv")
with (EXP_DIR / "routing_audit.json").open(encoding="utf-8") as f:
    audit = json.load(f)
print(f"WA margin thresholds: {audit['wa_thresholds']}")
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (family, sub) in zip(axes, summary.groupby("family")):
    native = sub[sub["ece_input"] == "native"].set_index("policy").reindex(config["policies"])
    ax.bar(native.index, native["rmse_mean"], yerr=native["rmse_std"], capsize=3)
    ax.set_title(f"{family} (native-missing ECE)")
    ax.set_ylabel("pooled RMSE (m3/m3)")
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
(EXP_DIR / "figures").mkdir(exist_ok=True)
fig.savefig(EXP_DIR / "figures" / "policy_rmse_by_family.png", dpi=150)
print("Saved figures/policy_rmse_by_family.png")


WA margin thresholds: {'v0': 1.9559823212899237, 'backbone': 1.5480887296641908}
                  family ece_input             policy  rmse_mean  rmse_std  mae_mean  bias_mean  ubrmse_mean    r2_mean  pearson_mean  rmse_change_vs_as_routed
   Clustering_V0_Full_k2      zero          as_routed   0.117805  0.001259  0.093832   0.070703     0.094226  -5.269953     -0.486731                  0.000000
   Clustering_V0_Full_k2      zero            c0_only   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965      0.141216                 -0.066460
   Clustering_V0_Full_k2      zero            c1_only   0.155617  0.001981  0.148323   0.148323     0.047076  -9.941304      0.132800                  0.037812
   Clustering_V0_Full_k2      zero    gapi_transplant   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965      0.141216                 -0.066460
   Clustering_V0_Full_k2      zero dynamic_transplant   0.051345  0.000215  0.044674   0.019798     0.047366  -0.190965

Saved figures/policy_rmse_by_family.png


## 4. Station-level verdict and stop rule
If `c0_only` does not recover toward the dynamic-router level (`RMSE ~0.05`),
the experts themselves are untransferable and router surgery stops here.


In [4]:
import pandas as pd

station = pd.read_csv(EXP_DIR / "station_metrics.csv")
pivot = station.groupby(["family", "ece_input", "policy", "station"], sort=False)["rmse"].mean().unstack("station")
print(pivot.to_string(float_format=lambda v: f"{v:.6f}"))
native = pivot.xs("native", level="ece_input")
for family in config["families"]:
    base = native.loc[(family, "as_routed")].mean()
    fixed = native.loc[(family, "c0_only")].mean()
    print(f"{family}: as_routed mean-station RMSE={base:.6f} -> c0_only={fixed:.6f} "
          f"(delta={fixed - base:+.6f})")


station                                                ECE_BBG_Lost_Meadow  ECE_BBG_Main_St  ECE_Renton_Garden_North  ECE_Renton_Garden_Shed  ECE_Renton_Home
family                   ece_input policy                                                                                                                    
Clustering_V0_Full_k2    zero      as_routed                      0.163857         0.048967                 0.061687                0.028708         0.188440
                                   c0_only                        0.036338         0.035448                 0.063735                0.022692         0.077614
                                   c1_only                        0.163857         0.170277                 0.070456                0.145785         0.197529
                                   gapi_transplant                0.036338         0.035448                 0.063735                0.022692         0.077614
                                   dynamic_transplan